In [ ]:
# This public repository version reads input files from the local repository.


In [ ]:
from pathlib import Path
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "outputs" / "oof" / "results").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open Jupyter from this repository, with outputs/oof/results available")
OOF_RESULTS_DIR = REPO_ROOT / "outputs" / "oof" / "results"
OOF_PLOT_DIR = REPO_ROOT / "outputs" / "oof" / "regenerated_figures"
OOF_PLOT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# Table 24 — Repeated 20×5 OOF, Real-only subset
# ============================================================

df = pd.DataFrame({
    "Model": [
        "BART",
        "T5",
        "SciBERT",
        "RoBERTa",
        "DeepSeek",
        "LLaMA3"
    ],
    "Low Recall": [
        0.9770,
        0.9541,
        0.9508,
        0.9672,
        0.9508,
        0.9213
    ],
    "High Recall": [
        0.8913,
        0.8913,
        0.8261,
        0.8913,
        0.8261,
        0.8043
    ]
})

# Δ = High - Low
df["Delta"] = df["High Recall"] - df["Low Recall"]

# ============================================================
# Paper-ready styling
# ============================================================

sns.set_theme(
    style="whitegrid",
    context="paper",
    font_scale=1.0
)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

fig, ax = plt.subplots(figsize=(7, 5))

# y positions
y = range(len(df))

# Connecting lines
for i, row in df.iterrows():
    ax.plot(
        [row["High Recall"], row["Low Recall"]],
        [i, i],
        linewidth=1.5,
        zorder=1
    )

# Points
sns.scatterplot(
    data=df,
    x="Low Recall",
    y="Model",
    s=75,
    marker="o",
    label="Low pressure",
    ax=ax,
    zorder=3
)

sns.scatterplot(
    data=df,
    x="High Recall",
    y="Model",
    s=75,
    marker="s",
    label="High pressure",
    ax=ax,
    zorder=3
)

# Δ annotations
for i, row in df.iterrows():
    midpoint = (row["Low Recall"] + row["High Recall"]) / 2

    ax.text(
        midpoint,
        i - 0.18,
        rf"$\Delta$ = {row['Delta']:.4f}",
        ha="center",
        va="center",
        fontsize=9
    )

# Axis
ax.set_xlabel("Positive-class recall")
ax.set_ylabel("")

# Fixed x-axis to emphasize differences
ax.set_xlim(0.78, 1.00)
ax.set_xticks([
    0.80, 0.85, 0.90, 0.95, 1.00
])

# Put BART at top
ax.invert_yaxis()

# Clean appearance
sns.despine(ax=ax, left=True)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.6,
    alpha=0.5
)
ax.grid(axis="y", visible=False)

# Legend
ax.legend(
    frameon=False,
    loc="upper right"
)

plt.tight_layout()

plt.savefig(
    OOF_PLOT_DIR / "Table24_RealOnly_PositiveRecall_Dumbbell.pdf",
    bbox_inches="tight",
    dpi=300
)

"""plt.savefig(
    OOF_PLOT_DIR / "Table24_RealOnly_PositiveRecall_Dumbbell.png",
    bbox_inches="tight",
    dpi=300
)"""

plt.show()